In [2]:
from dotenv import load_dotenv
import os
 
load_dotenv()
 
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated, Literal
import random
from datetime import datetime, timedelta

c:\Users\USER\Desktop\code\Langgraph-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("API_TOKEN"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0.3
)

In [4]:
@tool
def track_order(order_id: str) -> str:
    """Track a customer's food order status. Args: order_id: Order ID like 'ORD-1001'"""
    orders = {
        "ORD-1001": {
            "items": "Jollof Rice + Chicken, Moi Moi, Chapman",
            "status": "🛵 Out for Delivery",
            "eta": "15 minutes",
            "rider": "Emeka",
            "rider_phone": "0812-345-6789",
            "total": "₦8,500"
        },
        "ORD-1002": {
            "items": "Pounded Yam + Egusi Soup, Suya (10 sticks)",
            "status": "👨‍🍳 Being Prepared",
            "eta": "35 minutes",
            "rider": "Not assigned yet",
            "rider_phone": "N/A",
            "total": "₦12,000"
        },
        "ORD-1003": {
            "items": "Fried Rice + Turkey, Pepper Soup, Zobo",
            "status": "✅ Delivered",
            "eta": "Delivered at 2:30 PM",
            "rider": "Chioma",
            "rider_phone": "0903-111-2222",
            "total": "₦9,200"
        },
        "ORD-1004": {
            "items": "Amala + Ewedu & Gbegiri, Assorted Meat",
            "status": "❌ Cancelled",
            "eta": "N/A",
            "rider": "N/A",
            "rider_phone": "N/A",
            "total": "₦7,800 (Refund processing)"
        },
    }
    data = orders.get(order_id.upper())
    if data:
        return f"""
Order: {order_id.upper()}
Items: {data['items']}
Status: {data['status']}
ETA: {data['eta']}
Rider: {data['rider']} ({data['rider_phone']})
Total: {data['total']}"""
    return f"Order {order_id} not found. Try ORD-1001, ORD-1002, ORD-1003, or ORD-1004."
 
 
@tool
def cancel_order(order_id: str, reason: str) -> str:
    """Cancel a customer's order. Only works if order is not yet delivered.
    Args: order_id: Order ID. reason: Why the customer wants to cancel."""
    non_cancellable = ["ORD-1003", "ORD-1004"]
    if order_id.upper() in non_cancellable:
        return f"Cannot cancel {order_id} — it's already delivered or previously cancelled."
    return f"✅ Order {order_id.upper()} has been cancelled. Reason: {reason}. Refund of full amount will reflect in 24-48 hours."
 
 
@tool
def report_issue(order_id: str, issue: str) -> str:
    """Report a problem with an order (wrong item, cold food, missing item, etc).
    Args: order_id: Order ID. issue: Description of the problem."""
    ticket_id = f"TKT-{random.randint(10000, 99999)}"
    return f"""
✅ Issue reported!
Ticket: {ticket_id}
Order: {order_id.upper()}
Issue: {issue}
Status: Under review — our team will respond within 1 hour.
Compensation: You may receive a discount code or free reorder."""
 

In [5]:
# ================================================================
# MENU & PRICING TOOLS
# ================================================================
 
@tool
def get_menu(category: str) -> str:
    """Get the menu for a food category. 
    Args: category: 'rice', 'swallow', 'soups', 'protein', 'drinks', 'snacks', or 'all'"""
    menu = {
        "rice": [
            ("Jollof Rice", "₦2,500", "Party-style smoky jollof with tomato base"),
            ("Fried Rice", "₦2,500", "Veggie fried rice with carrots, peas, liver"),
            ("Coconut Rice", "₦2,800", "Creamy coconut-infused rice"),
            ("Ofada Rice + Sauce", "₦3,000", "Local brown rice with spicy ofada sauce"),
            ("Native Jollof", "₦3,500", "Firewood jollof cooked the traditional way"),
        ],
        "swallow": [
            ("Pounded Yam", "₦2,000", "Smooth, stretchy pounded yam"),
            ("Amala", "₦1,500", "Silky yam flour amala"),
            ("Eba (Garri)", "₦1,200", "Classic garri swallow"),
            ("Semovita", "₦1,500", "Light semolina swallow"),
            ("Fufu", "₦1,500", "Fermented cassava fufu"),
        ],
        "soups": [
            ("Egusi Soup", "₦3,000", "Melon seed soup with spinach"),
            ("Efo Riro", "₦2,800", "Rich spinach stew, Yoruba-style"),
            ("Ogbono Soup", "₦2,800", "Draw soup with ogbono seeds"),
            ("Ewedu + Gbegiri", "₦2,500", "Jute leaf + bean soup combo"),
            ("Pepper Soup", "₦3,500", "Spicy assorted meat pepper soup"),
            ("Banga Soup", "₦3,200", "Palm fruit soup, Delta-style"),
        ],
        "protein": [
            ("Chicken (1 piece)", "₦2,000", "Peppered or fried chicken"),
            ("Turkey", "₦2,500", "Juicy peppered turkey"),
            ("Assorted Meat", "₦2,000", "Mix of beef, tripe, cow foot"),
            ("Grilled Fish", "₦3,000", "Whole tilapia or catfish"),
            ("Suya (10 sticks)", "₦3,500", "Spicy grilled beef skewers"),
            ("Peppered Goat Meat", "₦3,000", "Spicy goat meat chunks"),
        ],
        "drinks": [
            ("Zobo", "₦800", "Chilled hibiscus drink"),
            ("Chapman", "₦1,200", "Nigerian cocktail mocktail"),
            ("Palm Wine", "₦1,500", "Fresh sweet palm wine"),
            ("Kunu", "₦800", "Millet-based traditional drink"),
            ("Bottled Water", "₦200", "60cl bottled water"),
            ("Soft Drink (Can)", "₦500", "Coke, Fanta, Sprite"),
        ],
        "snacks": [
            ("Puff Puff (10 pcs)", "₦1,000", "Sweet fried dough balls"),
            ("Meat Pie", "₦800", "Flaky pastry with seasoned beef"),
            ("Chin Chin", "₦600", "Crunchy fried dough snack"),
            ("Moi Moi", "₦800", "Steamed bean pudding"),
            ("Akara (5 pcs)", "₦600", "Fried bean cakes"),
            ("Plantain (Dodo)", "₦700", "Fried ripe plantain"),
        ],
    }
    
    if category.lower() == "all":
        result = "🍛 MAMA NKECHI'S FULL MENU\n" + "=" * 40 + "\n"
        for cat, items in menu.items():
            result += f"\n📌 {cat.upper()}\n"
            for name, price, desc in items:
                result += f"  {name} — {price}\n    {desc}\n"
        return result
    
    items = menu.get(category.lower())
    if items:
        result = f"📌 {category.upper()} MENU\n"
        for name, price, desc in items:
            result += f"\n  {name} — {price}\n    {desc}\n"
        return result
    return f"Category '{category}' not found. Try: rice, swallow, soups, protein, drinks, snacks, or all."
 
 
@tool
def check_availability(item: str) -> str:
    """Check if a specific menu item is available right now.
    Args: item: Name of the food item (e.g., 'Jollof Rice', 'Suya')"""
    unavailable = ["palm wine", "native jollof", "banga soup"]
    if item.lower() in unavailable:
        return f"❌ Sorry, {item} is currently UNAVAILABLE today. We'll have it back tomorrow!"
    return f"✅ {item} is AVAILABLE! Ready in 20-30 minutes after ordering."
 
 
@tool
def get_delivery_areas() -> str:
    """Get the list of areas we deliver to and delivery fees."""
    return """
🛵 DELIVERY AREAS & FEES
 
Lagos Mainland:
  Yaba, Surulere, Ikeja, Maryland — ₦1,000
  Ogba, Agege, Mushin, Oshodi — ₦1,200
  Festac, Amuwo-Odofin — ₦1,500
 
Lagos Island:
  Victoria Island, Lekki Phase 1 — ₦1,500
  Ikoyi, Lagos Island — ₦1,200
  Ajah, Lekki Phase 2 — ₦2,000
 
Free delivery on orders above ₦15,000!
Delivery time: 30-60 minutes depending on location."""

In [6]:
# ================================================================
# GENERAL SUPPORT TOOLS
# ================================================================
 
@tool
def get_business_hours() -> str:
    """Get Mama Nkechi's business hours and contact info."""
    return """
🕐 BUSINESS HOURS
  Monday - Saturday: 10:00 AM - 10:00 PM
  Sunday: 12:00 PM - 9:00 PM
  Public Holidays: 12:00 PM - 8:00 PM
 
📞 CONTACT
  WhatsApp: 0812-MAMA-FOOD (0812-6262-3663)
  Instagram: @mamankechi_kitchen
  Phone: 0901-234-5678
  Email: hello@mamankechi.com
 
📍 PICKUP LOCATION
  15 Awolowo Road, Ikoyi, Lagos"""
 
 
@tool
def search_faq(question: str) -> str:
    """Search frequently asked questions. Args: question: Customer's question"""
    faqs = {
        "refund": "Refunds take 24-48 hours to process. For cancelled orders, you get a full refund. For quality issues, you may get a refund or free reorder.",
        "payment": "We accept: Bank Transfer, Card Payment, USSD, Cash on Delivery. For card issues, try another card or use transfer.",
        "catering": "We do catering for events! Minimum order: 50 persons. Contact us on WhatsApp (0812-6262-3663) for catering packages and pricing.",
        "allergen": "Please inform us of any allergies when ordering. Our food may contain: groundnuts, crayfish, palm oil, gluten. Ask for allergen-free options.",
        "bulk": "Bulk orders (10+ plates) get 10% discount. Party jollof packages start from ₦25,000 for 20 persons.",
        "loyalty": "Join our Mama's VIP Club! Every 10th order is FREE. Download our app or ask for your loyalty card at pickup.",
    }
    q_lower = question.lower()
    for keyword, answer in faqs.items():
        if keyword in q_lower:
            return f"FAQ: {answer}"
    return "No FAQ match. Our support team will help you directly. You can also WhatsApp us at 0812-6262-3663."
 
 
@tool
def leave_feedback(rating: int, comment: str) -> str:
    """Submit customer feedback. Args: rating: 1-5 stars. comment: Feedback text."""
    if rating < 1 or rating > 5:
        return "Rating must be 1-5 stars."
    stars = "⭐" * rating
    return f"""
✅ Feedback submitted! Thank you!
Rating: {stars} ({rating}/5)
Comment: {comment}
 
{'🎉 We are glad you enjoyed it!' if rating >= 4 else '🙏 Sorry about your experience. Our manager will reach out within 24 hours.'}"""

In [7]:
order_tools = [track_order, cancel_order, report_issue]
menu_tools = [get_menu, check_availability, get_delivery_areas]
general_tools = [get_business_hours, search_faq, leave_feedback]
 
order_llm = llm.bind_tools(order_tools)
menu_llm = llm.bind_tools(menu_tools)
general_llm = llm.bind_tools(general_tools)
 
all_tools = order_tools + menu_tools + general_tools
tools_by_name = {t.name: t for t in all_tools}

In [8]:
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    category: str
    step_count: int

In [9]:
BRAND_PERSONA = """You are the friendly customer support assistant for **Mama Nkechi's Kitchen** 🍛, 
a popular Nigerian food delivery brand in Lagos. 
 
Your personality:
- Warm, friendly, and proudly Nigerian
- Use light Nigerian English/pidgin occasionally (e.g., "No wahala!", "E go be alright")
- Always helpful and empathetic
- Proud of the food — you know every dish on the menu
- Address customers respectfully"""
 
 
def classify_query(state: SupportState) -> dict:
    """Classify query into orders, menu, or general."""
    prompt = SystemMessage(content="""Classify the customer's message into EXACTLY one word:
    - "orders" — tracking orders, cancellation, complaints about delivery, missing items, wrong food
    - "menu" — food menu, prices, what's available, delivery areas, what do you sell
    - "general" — business hours, contact info, FAQs, feedback, catering, payment methods, loyalty program
    Respond with ONLY the category word.""")
    
    response = llm.invoke([prompt] + state["messages"])
    category = response.content.strip().lower()
    if category not in ["orders", "menu", "general"]:
        category = "general"
    return {"category": category}
 
 
def order_agent(state: SupportState) -> dict:
    """Handles order tracking, cancellations, and complaints."""
    system = SystemMessage(content=f"""{BRAND_PERSONA}
    
    You are the ORDERS specialist. You handle:
    - Order tracking (use track_order tool)
    - Order cancellation (use cancel_order tool)
    - Reporting issues with orders (use report_issue tool)
    
    Always use tools to look up real data. If the customer doesn't give an order ID, ask for it politely.""")
    
    response = order_llm.invoke([system] + state["messages"])
    return {"messages": [response], "step_count": state.get("step_count", 0) + 1}
 
 
def menu_agent(state: SupportState) -> dict:
    """Handles menu inquiries, pricing, and availability."""
    system = SystemMessage(content=f"""{BRAND_PERSONA}
    
    You are the MENU & PRICING specialist. You handle:
    - Showing the menu (use get_menu tool with categories: rice, swallow, soups, protein, drinks, snacks, all)
    - Checking if items are available (use check_availability tool)
    - Delivery areas and fees (use get_delivery_areas tool)
    
    Be enthusiastic about the food! Suggest popular combos like:
    - "Jollof Rice + Chicken + Chapman" (classic combo)
    - "Pounded Yam + Egusi Soup + Assorted Meat" (swallow lovers)
    - "Suya + Zobo" (perfect evening snack)""")
    
    response = menu_llm.invoke([system] + state["messages"])
    return {"messages": [response], "step_count": state.get("step_count", 0) + 1}
 
 
def general_agent(state: SupportState) -> dict:
    """Handles general inquiries, FAQs, and feedback."""
    system = SystemMessage(content=f"""{BRAND_PERSONA}
    
    You are the GENERAL SUPPORT specialist. You handle:
    - Business hours and contact info (use get_business_hours tool)
    - FAQs (use search_faq tool)
    - Customer feedback (use leave_feedback tool)
    
    For questions about ordering, suggest our WhatsApp: 0812-6262-3663""")
    
    response = general_llm.invoke([system] + state["messages"])
    return {"messages": [response], "step_count": state.get("step_count", 0) + 1}
 
 
def execute_tools(state: SupportState) -> dict:
    """Run any tools the specialist requested."""
    results = []
    last_msg = state["messages"][-1]
    for tc in last_msg.tool_calls:
        try:
            result = tools_by_name[tc["name"]].invoke(tc["args"])
            results.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
        except Exception as e:
            results.append(ToolMessage(content=f"Error: {e}", tool_call_id=tc["id"]))
    return {"messages": results}

In [10]:
def route_to_specialist(state: SupportState) -> str:
    return state["category"]
 
def after_specialist(state: SupportState) -> Literal["tools", "__end__"]:
    if state.get("step_count", 0) >= 10:
        return "__end__"
    last = state["messages"][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return "tools"
    return "__end__"
 
def after_tools(state: SupportState) -> str:
    return state["category"]

In [11]:
builder = StateGraph(SupportState)
 
builder.add_node("classify", classify_query)
builder.add_node("orders", order_agent)
builder.add_node("menu", menu_agent)
builder.add_node("general", general_agent)
builder.add_node("tools", execute_tools)
 
builder.add_edge(START, "classify")
 
builder.add_conditional_edges("classify", route_to_specialist, {
    "orders": "orders", "menu": "menu", "general": "general"
})
 
builder.add_conditional_edges("orders", after_specialist, ["tools", "__end__"])
builder.add_conditional_edges("menu", after_specialist, ["tools", "__end__"])
builder.add_conditional_edges("general", after_specialist, ["tools", "__end__"])
 
builder.add_conditional_edges("tools", after_tools, ["orders", "menu", "general"])
 
memory = InMemorySaver()
bot = builder.compile(checkpointer=memory)

In [14]:
def respond(message: str, history: list) -> str:
    """Process a message through the LangGraph bot and return the response."""
    
    config = {"configurable": {"thread_id": "gradio_session"}}
    
    try:
        result = bot.invoke(
            {"messages": [HumanMessage(content=message)], "category": "", "step_count": 0},
            config=config
        )
        
        # Find the last AI message with actual content
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        
        return "I'm looking into that for you... Please try again!"
    
    except Exception as e:
        return f"Sorry, I encountered an error: {str(e)}\nPlease try again or WhatsApp us at 0812-6262-3663."
 
 
# Build the Gradio interface
demo = gr.ChatInterface(
    fn=respond,
    title="🍛 Mama Nkechi's Kitchen — Customer Support",
    description="Welcome to Mama Nkechi's Kitchen! 🇳🇬 I can help with orders, menu & pricing, or general questions.",
    examples=[
        "What's on your rice menu?",
        "Track my order ORD-1001",
        "Is Jollof Rice available?",
        "Where do you deliver to in Lagos?",
        "What are your business hours?",
        "Show me your full menu",
    ],
)

if __name__ == "__main__":
    demo.launch(share=True)
 


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
